# Climate Claim Scanner — Daily GPU Maintenance

**In-house maintenance pass. Run 2–3×/week (or daily).** Local ingestion runs
autonomously every 24h on CPU and accumulates raw posts; this notebook does the
GPU-heavy stages and exports results back for the website:

`classify → vision → reindex → evaluate` (see `climate_verifier/maintenance.py`).

Each stage writes a health heartbeat (`data/health.json`) and evaluation appends a
drift snapshot (`data/eval_history.jsonl`) the app plots. Set `DRIVE_DIR` below to
your Google Drive sync folder.


## 1 · Confirm GPU


In [ ]:
!nvidia-smi -L

## 2 · Clone / update the repo (public — no token needed)


In [ ]:
import os
REPO   = "https://github.com/Sadaf-Burhan/ClimateClaimVerifier.git"
BRANCH = "multimodal-edge-gating"
if not os.path.exists("ClimateClaimVerifier"):
    !git clone -b {BRANCH} {REPO}
%cd ClimateClaimVerifier
!git pull origin {BRANCH}

## 3 · Install the package + dependencies


In [ ]:
!pip install -e . -q
print("installed")

## 4 · Install Ollama + pull the models
`qwen2.5:3b` = classifier, `qwen2.5vl:7b` = vision (GPU-only).


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull qwen2.5:3b
!ollama pull qwen2.5vl:7b

## 5 · Pull the latest ingested DB from Drive
The local daily job writes `data/ingested.db`; sync it here so we classify the
freshest posts. **Edit `DRIVE_DIR`** to your synced folder.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/ClimateScanner"   # <-- your Drive folder
os.makedirs("data", exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
src = os.path.join(DRIVE_DIR, "ingested.db")
if os.path.exists(src):
    shutil.copy(src, "data/ingested.db")
    print("pulled latest ingested.db from Drive")
else:
    print("No ingested.db in Drive yet — using whatever is in the repo/data.")

## 6 · Run the maintenance chain
`--all` = classify → vision → reindex → evaluate. One broken stage doesn't block the
rest; failures are recorded in `data/health.json` and the cell exits non-zero.


In [ ]:
!python -m climate_verifier.maintenance --all

## 7 · Export results back to Drive (for the website)


In [ ]:
import os, shutil
for f in ["ingested.db", "eval_history.jsonl", "health.json"]:
    p = os.path.join("data", f)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(DRIVE_DIR, f))
        print("exported", f)
if os.path.exists("data/chroma_evidence"):
    shutil.copytree("data/chroma_evidence", os.path.join(DRIVE_DIR, "chroma_evidence"), dirs_exist_ok=True)
    print("exported chroma_evidence/")
print("\nDone. Pull these into the project dir; the app reads DB + eval_history + health.")

## 8 · Quick look — latest drift snapshot


In [ ]:
import json, pathlib
p = pathlib.Path("data/eval_history.jsonl")
if p.exists():
    print("latest eval:", json.loads(p.read_text().strip().splitlines()[-1]))
else:
    print("no eval history yet")